In [ ]:
#@title 1. Install Dependencies
# This cell installs the required libraries for the OCR process.
!pip install -q deepseek-ocr pdf2image
!apt-get install -q poppler-utils

In [ ]:
#@title 2. Upload PDF, Set Parameters, and Run OCR
from google.colab import files
import os
from pdf2image import convert_from_path
from deepseek_ocr.ocr import OCR

# --- 1. Upload File ---
print("Please upload your PDF file.")
uploaded = files.upload()

pdf_path = ""
if uploaded:
    pdf_path = list(uploaded.keys())[0]
    print(f"Successfully uploaded '{pdf_path}'")
else:
    print("No file uploaded. Halting execution.")

# --- Form for parameters ---
#@markdown ---
#@markdown ### 2. Set Page Range (optional)
#@markdown If you leave the pages as is (start 1, end 0), the entire document will be processed.
start_page = 1 #@param {type:"number"}
end_page = 0 #@param {type:"number"}

def run_ocr(pdf_path, start_page, end_page):
    """
    Main function to handle the OCR process.
    """
    print("\nStarting OCR process...")

    # Validate file path
    if not os.path.exists(pdf_path):
        print(f"Error: File not found at '{pdf_path}'")
        return

    # Validate page numbers
    if end_page != 0 and end_page < start_page:
        print(f"Error: End page ({end_page}) cannot be smaller than start page ({start_page}).")
        return

    # --- Convert PDF to Images ---
    print("Converting PDF pages to images...")
    first_page_to_process = start_page
    last_page_to_process = end_page if end_page > 0 else None
    
    try:
        images = convert_from_path(pdf_path, first_page=first_page_to_process, last_page=last_page_to_process, dpi=300)
        print(f"Successfully converted {len(images)} pages.")
    except Exception as e:
        print(f"Failed to convert PDF to images. Error: {e}")
        return

    # --- Perform OCR ---
    print("Initializing OCR model...")
    ocr_engine = OCR()
    full_text = ""
    print("Model initialized. Starting page-by-page OCR.")

    for i, image in enumerate(images):
        current_page_num = first_page_to_process + i
        print(f"Processing page {current_page_num}...")
        try:
            # ocr() returns a tuple of (text, bounding_boxes)
            text, _ = ocr_engine.ocr(image)
            full_text += f"--- Page {current_page_num} ---\n{text}\n\n"
        except Exception as e:
            print(f"An error occurred while OCRing page {current_page_num}: {e}")

    # --- Save and Download Output ---
    output_filename = f"{os.path.splitext(os.path.basename(pdf_path))[0]}_ocr_output.txt"
    try:
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(full_text)
        print(f"\nOCR complete. Text saved to '{output_filename}'.")
        files.download(output_filename)
        print(f"'{output_filename}' has been downloaded.")
    except Exception as e:
        print(f"Failed to save or download the file. Error: {e}")

# --- Execute if a file was uploaded ---
if pdf_path:
    run_ocr(pdf_path, start_page=start_page, end_page=end_page)